# Power BI Semantic Model Security Audit — Fabric Notebook

This notebook harvests everything needed to answer five access-audit questions:

1. **Which users** have access to each semantic model?
2. **Which security role** is each user tagged to?
3. **Which AD (Entra ID) group** grants that access?
4. **What workspace role** do they hold?
5. **What DAX filters (RLS)** — and table/column restrictions (OLS) — does each role apply?

## 🔎 Running in DataFrame mode (no tables created)

**Table creation is commented out.** Each step builds a Spark **DataFrame**, registers it as a **temp view** so later
cells can still query it, and `display()`s it so you can eyeball the data. **Nothing is written to the lakehouse.**

| Step | DataFrame | Temp view | Answers |
|---|---|---|---|
| 1 | `df_model_details` | `pbi_model_details` | audit scope |
| 2 | `df_security_audit` | `pbi_model_security_audit_details` | Q5 — RLS + OLS |
| 3 | `df_role_members` | `pbi_model_security_role_association_details` | Q2/Q3 — role → AD group |
| 4 | `df_adgroup_users` | `pbi_model_security_adgroup_user_details` | Q1/Q3 — group → people |
| 5 | `df_workspace_access` | `pbi_workspace_access_audit_details` | Q4 — workspace roles |

**To land the data for modelling** (both optional, inside `publish()`): **Option A** — persist as Delta **tables** (also set
`SRC = f"{LH}."`) for a production model that imports / Direct Lakes them; or **Option B** — export each result as a **CSV**,
the very files the downloadable sample report's Power Query reads. Default writes nothing (DataFrame + temp view only).

**Self-contained — no external files, no SharePoint.** Fill in the **CONFIG** cell below (app credentials +
the list of models to audit) and run top to bottom. The scope is a small DataFrame you type directly, so there is
**no SharePoint, no Excel, no `config.json` and no `lakehouse_utils.py`** — just paste your client id/secret and go.

> **Prerequisites:**
> - An Entra **app registration (service principal)** with Graph **`Group.Read.All`** (application permission, admin-consented) — used in step 4 to expand AD groups to users.
> - The SP added to **every audited workspace** (Contributor).
> - Tenant settings: *Service principals can use Fabric APIs*; **XMLA endpoint** = Read on the capacity.
> - `%pip install semantic-link-labs` (or add it to your Fabric environment).

## 0a · Install libraries (run first, then restart the kernel)

`set_service_principal` — used to read each model's security as the service principal — needs **semantic-link ≥ 0.12.0**.
The default Fabric runtime often ships an older build, which fails with *"cannot import name 'set_service_principal'."*
Run the next cell, **restart the kernel** (or hit *Run all* again), then continue.

In [ ]:
%pip install -U semantic-link semantic-link-labs
# >>> After this finishes: RESTART THE KERNEL, then run the notebook from the top. <<<

## 0b · Config & setup

Fill in the **CONFIG** cell (credentials + the models to audit), then run the setup cell.

- **Auth** — one MSAL app + a `headers_for(api)` helper (`graph` / `powerbi` / `fabric`).
- **`publish(df, name)`** — the single output point: registers a temp view, plus two commented "land it" options —
  **Delta table** (production) or **CSV export** (portable, for the sample report). Default writes nothing.

> **Secret hygiene.** Pasting a secret inline is fine for a quick **test**. For anything real, read it from
> **Azure Key Vault** with `notebookutils.credentials.getSecret(...)` and don't share the filled-in notebook.

In [ ]:
# ============================================================
# CONFIG  —  fill these in, then run the whole notebook.
# (Self-contained: no config.json, no lakehouse_utils.py needed.)
# ============================================================
TENANT_ID     = "<your-entra-tenant-id>"
CLIENT_ID     = "<your-app-registration-client-id>"
CLIENT_SECRET = "<your-client-secret>"

# --- Models to audit: list them directly (workspace name, semantic model name) ---
# Add as many rows as you like. No SharePoint, no Excel, no file reading.
MODELS = [
    ("<workspace name>", "<semantic model name>"),
    # ("<workspace name>", "<another semantic model>"),
]

# Worked example in section 8 (blank -> first model in MODELS)
EXAMPLE_MODEL = ""

print(f"Config set: {len(MODELS)} model(s) to audit. Edit the placeholders above before running.")

In [ ]:
import sempy
print("semantic-link:", sempy.__version__, "(need >= 0.12.0 for set_service_principal)")

import msal, requests, json, time
import pandas as pd
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType

# (TENANT_ID / CLIENT_ID / CLIENT_SECRET come from the CONFIG cell above.)

# --- One MSAL app + one token helper per API scope -------------------------
_app = msal.ConfidentialClientApplication(
    CLIENT_ID,
    authority=f"https://login.microsoftonline.com/{TENANT_ID}",
    client_credential=CLIENT_SECRET,
)

_SCOPES = {
    "graph":   "https://graph.microsoft.com/.default",
    "powerbi": "https://analysis.windows.net/powerbi/api/.default",
    "fabric":  "https://api.fabric.microsoft.com/.default",
}

def headers_for(api, content_type=False):
    """Authorization header dict for 'graph' | 'powerbi' | 'fabric'."""
    token = _app.acquire_token_for_client(scopes=[_SCOPES[api]])["access_token"]
    h = {"Authorization": f"Bearer {token}"}
    if content_type:
        h["Content-Type"] = "application/json"
    return h

GRAPH  = "https://graph.microsoft.com/v1.0"
PBI    = "https://api.powerbi.com/v1.0/myorg"
FABRIC = "https://api.fabric.microsoft.com"

# ---------------------------------------------------------------------------
# OUTPUT MODE — table creation is COMMENTED OUT for now.
#   SRC = ""          -> later cells query the TEMP VIEWS (current mode)
#   SRC = f"{LH}."    -> later cells query the real Delta TABLES
# ---------------------------------------------------------------------------
LH  = "pbi_security_audit.dbo"   # lakehouse schema (only used once you enable writes)
SRC = ""                          # <-- keep "" while table creation is commented out


def publish(df, name):
    """Point everything downstream at the DataFrame, and (optionally) land it for modelling.

    Registers 'df' as a temp view called 'name' so later SQL cells work unchanged.
    Two ways to LAND the data for the semantic model - both commented out
    (default = DataFrame + temp view only, nothing written):

      Option A - persist as a Delta TABLE   -> production: the model imports / Direct Lakes the tables.
                 (also set SRC = f"{LH}." in cell 0 so later cells read the tables.)
      Option B - export as a CSV FILE        -> portable: point a model at the CSVs, exactly like the
                 downloadable sample report (its Power Query reads these same CSVs).
    """
    df.createOrReplaceTempView(name)
    print(f"  [{name}] {df.count()} rows -> temp view (nothing written)")

    # --- Option A: persist as a Delta table (production) ----------------------
    # (
    #     df.write.format("delta").mode("overwrite")
    #       .option("overwriteSchema", "true")
    #       .saveAsTable(f"{LH}.{name}")
    # )

    # --- Option B: export as a CSV for modelling (portable) -------------------
    # import os
    # os.makedirs("/lakehouse/default/Files/audit_csv", exist_ok=True)
    # df.toPandas().to_csv(f"/lakehouse/default/Files/audit_csv/{name}.csv", index=False)

    return df


print("Setup complete. Mode: DataFrame + temp views (table creation commented out).")

## 1 · Build the audit scope

**Logic:** the audit *scope* is just the list of models you typed in **`MODELS`** (CONFIG cell). We turn it into
**`df_model_details`** (temp view `pbi_model_details`) — the driver DataFrame steps 2–5 read. No files, no SharePoint.

> 💡 `persist=False` keeps everything DataFrame-only (no tables written).

In [ ]:
# Build the scope DataFrame directly from MODELS (no file reads, no SharePoint)
df_model_details = spark.createDataFrame(
    [(w, m) for (w, m) in MODELS],
    ["Workspace_Name", "Semantic_Model_Name"],
)
df_model_details = publish(df_model_details, "pbi_model_details")
display(df_model_details)

## 2 · Extract RLS and OLS with semantic-link-labs (TOM)

**Logic — the heart of the solution.** For every model in scope we open a **read-only Tabular Object Model**
connection under the service principal and walk each role:

- **RLS:** every `TablePermission.FilterExpression` is the role's DAX filter on that table.
- **OLS:** `MetadataPermission` on tables/columns. A whole table hidden (`Table_Permission = "None"`) becomes one
  collapsed row (`Column_Name = "*"`); otherwise we emit one row per column with its **effective visibility**
  (`Hidden`/`Visible`), defaulting unlisted columns to `Read` — so the output answers "what does this role *see*?"
  without the reader needing to know OLS semantics.

Failures are recorded as **`Filter_Type = "ERROR"`** rows instead of crashing the run, so a workspace the service
principal can't read becomes a visible finding rather than a silent gap.

**Output:** `df_security_audit` (temp view `pbi_model_security_audit_details`).

In [ ]:
from sempy.fabric import set_service_principal
from sempy_labs.tom import connect_semantic_model

with set_service_principal(
    tenant_id=TENANT_ID, client_id=CLIENT_ID, client_secret=CLIENT_SECRET,
):
    model_rows = spark.sql(f"""
        SELECT DISTINCT Workspace_Name, Semantic_Model_Name
        FROM {SRC}pbi_model_details
    """).collect()

    security_rows = []
    print(f"Starting security audit for {len(model_rows)} models...")

    for r in model_rows:
        workspace, dataset = r["Workspace_Name"], r["Semantic_Model_Name"]
        try:
            with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:

                for role in tom.model.Roles:
                    role_name = str(role.Name)

                    # Role -> OLS lookup: {table: {permission, columns{col: permission}}}
                    role_ols_map = {
                        str(tp.Table.Name): {
                            "Table_Permission": str(tp.MetadataPermission),
                            "Columns": {
                                str(cp.Column.Name): str(cp.MetadataPermission)
                                for cp in tp.ColumnPermissions
                            },
                        }
                        for tp in role.TablePermissions
                    }

                    # ---- PART A: RLS filter expressions -----------------------
                    for perm in role.TablePermissions:
                        expr = perm.FilterExpression
                        if expr and str(expr).strip():
                            security_rows.append(Row(
                                Workspace=workspace, Semantic_Model=dataset,
                                Role_Name=role_name, Table_Name=str(perm.Table.Name),
                                Column_Name=None, Filter=str(expr).strip(), Filter_Type="RLS",
                                OLS_Level=None, Column_Permission=None,
                                Effective_Visibility=None, Error_Message=None,
                            ))

                    # ---- PART B: OLS visibility per table/column -------------
                    for table in tom.model.Tables:
                        table_name = str(table.Name)
                        sec = role_ols_map.get(table_name, {"Table_Permission": "Read", "Columns": {}})

                        # Case 1: whole table hidden -> one collapsed row
                        if sec["Table_Permission"] == "None":
                            security_rows.append(Row(
                                Workspace=workspace, Semantic_Model=dataset,
                                Role_Name=role_name, Table_Name=table_name,
                                Column_Name="*", Filter=None, Filter_Type="OLS",
                                OLS_Level="Table", Column_Permission="None",
                                Effective_Visibility="Hidden", Error_Message=None,
                            ))
                            continue

                        # Case 2: table readable -> check each column
                        for column in table.Columns:
                            if str(column.Type) == "RowNumber":
                                continue
                            col = str(column.Name)
                            col_perm = sec["Columns"].get(col, "Read")
                            security_rows.append(Row(
                                Workspace=workspace, Semantic_Model=dataset,
                                Role_Name=role_name, Table_Name=table_name,
                                Column_Name=col, Filter=None, Filter_Type="OLS",
                                OLS_Level="Column" if col in sec["Columns"] else "None",
                                Column_Permission=col_perm,
                                Effective_Visibility="Hidden" if col_perm == "None" else "Visible",
                                Error_Message=None,
                            ))

        except Exception as e:
            print(f"Error processing {dataset}: {e} "
                  "— check that the service principal is added to the workspace")
            security_rows.append(Row(
                Workspace=workspace, Semantic_Model=dataset, Role_Name=None,
                Table_Name=None, Column_Name=None, Filter=None,
                Filter_Type="ERROR", OLS_Level=None, Column_Permission=None,
                Effective_Visibility=None, Error_Message=str(e),
            ))

    schema = StructType([
        StructField(c, StringType(), True) for c in [
            "Workspace", "Semantic_Model", "Role_Name", "Table_Name",
            "Column_Name", "Filter", "Filter_Type", "OLS_Level",
            "Column_Permission", "Effective_Visibility", "Error_Message",
        ]
    ])

    df_security_audit = publish(
        spark.createDataFrame(security_rows, schema=schema),
        "pbi_model_security_audit_details",
    )

display(df_security_audit)

## 3 · Role memberships (and the query scale-out trap)

**Logic:** role *definitions* are only half the story — we also need role *membership* (which Entra group or user is
tagged to each role). TOM exposes `role.Members`, and each member's `MemberID` is the **Entra object ID** — the join
key to the Graph data in step 4.

**The trap:** an earlier version used DAX `INFO.ROLES()` / `INFO.ROLEMEMBERSHIPS()`, which returns unreliable
membership on a **query scale-out** read replica. **The fix:** detect `queryScaleOutSettings` via REST; if scale-out
is on, set `maxReadOnlyReplicas = 0`, wait for the replicas to drain, extract via TOM on the primary, then
**restore the original setting** — in a `finally` block so it restores even if extraction fails.

**Output:** `df_role_members` (temp view `pbi_model_security_role_association_details`).

In [ ]:
headers = headers_for("powerbi", content_type=True)


def get_workspace_id(name):
    groups = requests.get(f"{PBI}/groups", headers=headers).json()["value"]
    return next((g["id"] for g in groups if g["name"] == name), None)

def get_dataset_id(ws_id, name):
    ds = requests.get(f"{PBI}/groups/{ws_id}/datasets", headers=headers).json()["value"]
    return next((d["id"] for d in ds if d["name"] == name), None)

def get_scaleout_settings(ws_id, ds_id):
    r = requests.get(f"{PBI}/groups/{ws_id}/datasets/{ds_id}", headers=headers)
    return r.json().get("queryScaleOutSettings", {})

def update_scaleout(ws_id, ds_id, settings):
    requests.patch(
        f"{PBI}/groups/{ws_id}/datasets/{ds_id}", headers=headers,
        data=json.dumps({"queryScaleOutSettings": settings}),
    ).raise_for_status()


model_rows = spark.sql(f"""
    SELECT DISTINCT Workspace_Name, Semantic_Model_Name
    FROM {SRC}pbi_model_details
""").collect()

all_results = []

for row in model_rows:
    workspace, dataset = row["Workspace_Name"], row["Semantic_Model_Name"]
    print(f"Processing: {workspace} | {dataset}")
    try:
        ws_id = get_workspace_id(workspace)
        ds_id = get_dataset_id(ws_id, dataset)
        if not ws_id or not ds_id:
            print("  Workspace or dataset not found — skipping.")
            continue

        original = get_scaleout_settings(ws_id, ds_id)
        scaleout_enabled = original.get("maxReadOnlyReplicas", 0) != 0

        try:
            if scaleout_enabled:
                print("  Scale-out enabled — disabling temporarily...")
                update_scaleout(ws_id, ds_id, {"maxReadOnlyReplicas": 0})
                time.sleep(300)  # allow replicas to sync out

            with set_service_principal(
                tenant_id=TENANT_ID, client_id=CLIENT_ID, client_secret=CLIENT_SECRET,
            ):
                with connect_semantic_model(workspace=workspace, dataset=dataset, readonly=True) as tom:
                    for role in tom.model.Roles:
                        for member in role.Members:
                            all_results.append({
                                "Workspace": workspace,
                                "Semantic_Model": dataset,
                                "Role_Name": str(role.Name),
                                "Member_ID": str(member.MemberID),   # Entra object ID
                            })
        finally:
            # ALWAYS restore, even if the extraction above failed
            if scaleout_enabled:
                print("  Restoring original scale-out settings...")
                update_scaleout(ws_id, ds_id, original)

    except Exception as e:
        print("  Error:", e)

if all_results:
    df_role_members = publish(
        spark.createDataFrame(pd.DataFrame(all_results)).distinct(),
        "pbi_model_security_role_association_details",
    )
    display(df_role_members)
else:
    print("No role members found — check the models actually define roles.")

## 4 · Expand AD groups into users via Microsoft Graph

**Logic:** roles map to **Entra ID groups**, not individuals. A naming convention (`pbi-sec-*`) makes the lookup one
filtered Graph call; a second pass pulls each group's user members.

**The linchpin:** the role member's `MemberID` (from TOM, step 3) and the group's `id` (from Graph, here) are the
**same Entra object ID**. That equality is what lets the report join *role → group → person* with no name matching.

*(`/members` returns direct members only; for nested groups use `/transitiveMembers`.)*

**Output:** `df_adgroup_users` (temp view `pbi_model_security_adgroup_user_details`).

In [ ]:
headers = headers_for("graph")

# 1. All security groups following the naming convention (paginated)
url = f"{GRAPH}/groups?$filter=startswith(displayName,'pbi-sec')&$select=id,displayName"
groups = []
while url:
    data = requests.get(url, headers=headers).json()
    groups.extend(data.get("value", []))
    url = data.get("@odata.nextLink")

print(f"Found {len(groups)} groups")

# 2. Members of each group (users only; skip nested SPs/devices)
rows = []
for g in groups:
    url = f"{GRAPH}/groups/{g['id']}/members?$select=id,displayName,userPrincipalName"
    while url:
        data = requests.get(url, headers=headers).json()
        rows += [
            {
                "Group_Name": g["displayName"],
                "Group_Object_ID": g["id"],
                "User_Name": m.get("displayName"),
                "User_Object_ID": m.get("id"),
                "User_Principal_Name": m.get("userPrincipalName"),
            }
            for m in data.get("value", [])
            if m.get("@odata.type") == "#microsoft.graph.user"
        ]
        url = data.get("@odata.nextLink")

if rows:
    df_adgroup_users = publish(spark.createDataFrame(rows), "pbi_model_security_adgroup_user_details")
    display(df_adgroup_users)
else:
    print("No groups matched 'pbi-sec' — adjust the naming filter above.")

## 5 · Workspace role assignments via the Fabric REST API

**Logic:** model-level security means nothing if someone is **workspace Admin** — an Admin bypasses RLS entirely.
`GET /v1/workspaces/{id}/roleAssignments` returns every principal (user, group, or service principal) with a
workspace role; where the principal is a group we expand it to people using the step-4 DataFrame.

**Output:** `df_workspace_access` (temp view `pbi_workspace_access_audit_details`).

In [ ]:
headers = headers_for("fabric")

# Workspaces come straight from the scope CSV (distinct) — one source of truth
WORKSPACE_NAMES = [
    r["Workspace_Name"]
    for r in spark.sql(f"SELECT DISTINCT Workspace_Name FROM {SRC}pbi_model_details").collect()
]

# 1. Resolve workspace names -> IDs
all_ws = requests.get(f"{FABRIC}/v1/workspaces", headers=headers).json()["value"]
lookup = {w["displayName"]: w["id"] for w in all_ws}
workspaces = [{"id": lookup[n], "name": n} for n in WORKSPACE_NAMES if n in lookup]

# 2. Role assignments per workspace
frames = []
for ws in workspaces:
    data = requests.get(
        f"{FABRIC}/v1/workspaces/{ws['id']}/roleAssignments", headers=headers
    ).json().get("value", [])
    if not data:
        continue
    df = pd.json_normalize(data).rename(columns={
        "principal.displayName": "Name",
        "principal.type": "Type",
        "principal.userDetails.userPrincipalName": "Email",
        "role": "Role",
    })
    df["Workspace"] = ws["name"]
    frames.append(df[["Workspace", "Name", "Email", "Type", "Role"]])

combined = pd.concat(frames, ignore_index=True)

# 3. Where the principal is a group, expand it to individual users (from step 4)
df_ad = spark.sql(f"""
    SELECT Group_Name, User_Principal_Name AS Group_Member_Email
    FROM {SRC}pbi_model_security_adgroup_user_details
""").toPandas()

combined = combined.merge(df_ad, how="left", left_on="Name", right_on="Group_Name").drop(columns=["Group_Name"])
combined["Email"] = combined["Group_Member_Email"].combine_first(combined["Email"])
combined = combined.drop(columns=["Group_Member_Email"])

df_workspace_access = publish(spark.createDataFrame(combined), "pbi_workspace_access_audit_details")
display(df_workspace_access)

## 6 · Optional extensions

Three further access paths, each one cell away from the same pattern. **Optional** and they need extra permissions:

- **App audiences** need the tenant setting *Service principals can access read-only admin APIs* (a plain workspace SP gets **403** on `admin/*`).
- **Direct dataset permissions** catch one-off Build/Read grants on the model itself.
- **Lineage** answers "which reports does this model feed?".

These are defined as functions so nothing runs unless you call them.

In [ ]:
# --- A. App audiences (Power BI Admin API) — needs read-only admin API tenant setting
def collect_app_audiences(workspace_id):
    admin_headers = headers_for("powerbi")
    apps = requests.get(f"{PBI}/admin/apps?$top=5000", headers=admin_headers).json().get("value", [])
    out = []
    for app_info in [a for a in apps if a.get("workspaceId") == workspace_id]:
        users = requests.get(
            f"{PBI}/admin/apps/{app_info['id']}/users", headers=admin_headers,
        ).json().get("value", [])
        out.extend(users)
    return out


# --- B. Direct dataset permissions (in dataset users but NOT workspace users)
def direct_dataset_grants(ws_id, ds_id):
    h = headers_for("powerbi")
    ws_users = requests.get(f"{PBI}/groups/{ws_id}/users", headers=h).json()["value"]
    ds_users = requests.get(f"{PBI}/groups/{ws_id}/datasets/{ds_id}/users", headers=h).json()["value"]
    ws_ids = {u.get("identifier") for u in ws_users}
    return [u for u in ds_users if u.get("identifier") not in ws_ids]


# --- C. Lineage: report -> semantic model
def report_lineage(workspace_name):
    import sempy.fabric as fabric
    ws_id = fabric.resolve_workspace_id(workspace_name)
    return fabric.list_reports(workspace=ws_id), fabric.list_datasets(workspace=ws_id)


print("Extension helpers defined (nothing executed).")

## 7 · Validate the output — is this correct?

The code only truly runs against a live tenant, so "correct" means: **the five DataFrames landed, they join cleanly,
and the five questions answer.** Run these checks after a run.

In [ ]:
# 7a. Every DataFrame exists and has rows
views = [
    "pbi_model_details",
    "pbi_model_security_audit_details",
    "pbi_model_security_role_association_details",
    "pbi_model_security_adgroup_user_details",
    "pbi_workspace_access_audit_details",
]
for v in views:
    try:
        n = spark.sql(f"SELECT COUNT(*) AS c FROM {SRC}{v}").collect()[0]["c"]
        print(f"{v:<48} {n:>6} rows   {'ok' if n > 0 else 'EMPTY — check the step that builds it'}")
    except Exception as e:
        print(f"{v:<48} MISSING — that step has not run yet")

In [ ]:
# 7b. Referential integrity — every role member (group) should resolve to an AD group
orphans = spark.sql(f"""
    SELECT r.Semantic_Model, r.Role_Name, r.Member_ID
    FROM {SRC}pbi_model_security_role_association_details r
    LEFT JOIN {SRC}pbi_model_security_adgroup_user_details u
      ON r.Member_ID = u.Group_Object_ID
    WHERE u.Group_Object_ID IS NULL
""")
print(f"Role members with no matching AD group: {orphans.count()}")
# Non-zero is expected ONLY where a member is an individual user (not a group),
# or a group falls outside the pbi-sec-* convention scanned in step 4.
display(orphans)

In [ ]:
# 7c. Coverage gaps are recorded as data, not silent failures
print("Models the service principal could NOT read (should be intentional):")
display(spark.sql(f"""
    SELECT Workspace, Semantic_Model, Error_Message
    FROM {SRC}pbi_model_security_audit_details
    WHERE Filter_Type = 'ERROR'
"""))

## 8 · Worked example — one real semantic model, end to end

Set `EXAMPLE_MODEL` in the **CONFIG** cell to one of your models (or leave it blank — it defaults to the first
model in scope). The four cells below answer all five audit questions for that single model — the "does it work?" test.

In [ ]:
# Default to the first model in scope when EXAMPLE_MODEL is blank in config
if not EXAMPLE_MODEL:
    EXAMPLE_MODEL = spark.sql(f"SELECT Semantic_Model_Name FROM {SRC}pbi_model_details LIMIT 1").collect()[0][0]

# Q1 + Q2 + Q3 — who can see EXAMPLE_MODEL, via which role and which AD group?
print(f"--- Who can see: {EXAMPLE_MODEL} ---")
display(spark.sql(f"""
    SELECT DISTINCT r.Workspace, r.Role_Name,
                    u.Group_Name, u.User_Name, u.User_Principal_Name
    FROM {SRC}pbi_model_security_role_association_details r
    JOIN {SRC}pbi_model_security_adgroup_user_details u
      ON r.Member_ID = u.Group_Object_ID
    WHERE r.Semantic_Model = '{EXAMPLE_MODEL}'
    ORDER BY r.Role_Name, u.User_Name
"""))

In [ ]:
# Q5 (RLS) — what DAX filter does each role on EXAMPLE_MODEL apply?
print(f"--- RLS filters on: {EXAMPLE_MODEL} ---")
display(spark.sql(f"""
    SELECT Role_Name, Table_Name, Filter
    FROM {SRC}pbi_model_security_audit_details
    WHERE Semantic_Model = '{EXAMPLE_MODEL}'
      AND Filter_Type = 'RLS'
    ORDER BY Role_Name, Table_Name
"""))

In [ ]:
# Q5 (OLS) — what can each role on EXAMPLE_MODEL NOT see?
print(f"--- Hidden objects (OLS) on: {EXAMPLE_MODEL} ---")
display(spark.sql(f"""
    SELECT Role_Name, Table_Name,
           CASE WHEN Column_Name = '*' THEN '(entire table)' ELSE Column_Name END AS Hidden_Object,
           OLS_Level
    FROM {SRC}pbi_model_security_audit_details
    WHERE Semantic_Model = '{EXAMPLE_MODEL}'
      AND Filter_Type = 'OLS'
      AND Effective_Visibility = 'Hidden'
    ORDER BY Role_Name, Table_Name
"""))

In [ ]:
# Q4 — workspace roles, and the classic gap: RLS-restricted users who are ALSO workspace Admin
print("--- Workspace roles ---")
display(spark.sql(f"""
    SELECT Workspace, Name, Type, Role
    FROM {SRC}pbi_workspace_access_audit_details
    ORDER BY Workspace, Role
"""))

print("--- ⚠️ RLS-restricted users who are also workspace Admin/Member (RLS does NOT apply to them) ---")
display(spark.sql(f"""
    SELECT DISTINCT w.Workspace, w.Name, w.Email, w.Role AS Workspace_Role,
                    r.Semantic_Model, r.Role_Name AS RLS_Role
    FROM {SRC}pbi_workspace_access_audit_details w
    JOIN {SRC}pbi_model_security_adgroup_user_details u
      ON w.Email = u.User_Principal_Name
    JOIN {SRC}pbi_model_security_role_association_details r
      ON u.Group_Object_ID = r.Member_ID
    WHERE w.Role IN ('Admin', 'Member')
    ORDER BY w.Workspace, w.Name
"""))

---

## Next steps — land the data, then model it

Happy with the output? Pick how to land it for the semantic model (both live in `publish()`, cell 0):

- **Production** — uncomment **Option A** (Delta table) and set `SRC = f"{LH}."`. The five tables land in the lakehouse;
  point an **import or Direct Lake** semantic model at them and build the six-page audit report (covered in the blog).
  Scope from an **Excel inventory** + credentials from **`config.json`** is the fuller pattern shown in the article.
- **Portable / quick** — uncomment **Option B** (CSV export). Each result becomes a CSV under `Files/audit_csv`; point a
  model at those CSVs — exactly what the **downloadable sample report** does (its Power Query reads these same files).

Then schedule this notebook + a semantic model refresh in a Fabric data pipeline so every answer carries a fresh timestamp.

*All workspace names, model names, group names, and IDs in this notebook are fictionalized. The patterns are real.*